# Statistical Tests on the Fairness Results

The fairness summary shows small average score differences per demographic signal. With a small dataset, small numbers can easily be noise. This notebook runs two checks per signal:

1. A paired Wilcoxon signed-rank test on (original_score, changed_score) pairs. We use the non-parametric version because the differences are not normal.
2. A bootstrap 95% confidence interval on the mean absolute difference.

If the Wilcoxon p-value is small and the bootstrap interval does not include zero, we have more reason to believe the model is actually reacting to the demographic change.

In [ ]:
!pip install scipy pandas numpy

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

np.random.seed(42)
os.makedirs("results", exist_ok=True)

In [ ]:
from google.colab import files
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

In [ ]:
fairness = pd.read_csv("fairness_comparison.csv")
print(fairness.shape)
display(fairness.head())

In [ ]:
def bootstrap_ci(values, n=2000, ci=0.95):
    values = np.asarray(values)
    means = []
    for _ in range(n):
        sample = np.random.choice(values, size=len(values), replace=True)
        means.append(sample.mean())
    low = np.percentile(means, (1 - ci) / 2 * 100)
    high = np.percentile(means, (1 + ci) / 2 * 100)
    return low, high

In [ ]:
results = []
for signal, group in fairness.groupby("changed_signal"):
    a = group["original_score"].values
    b = group["changed_score"].values
    try:
        stat, p = wilcoxon(a, b)
    except ValueError:
        stat, p = float("nan"), float("nan")
    abs_diff = group["absolute_difference"].values
    low, high = bootstrap_ci(abs_diff)
    results.append({
        "changed_signal": signal,
        "n_pairs": len(group),
        "mean_absolute_difference": abs_diff.mean(),
        "bootstrap_ci_low": low,
        "bootstrap_ci_high": high,
        "wilcoxon_statistic": stat,
        "wilcoxon_p_value": p,
    })

stats_df = pd.DataFrame(results)
display(stats_df)

In [ ]:
stats_df.to_csv("results/fairness_statistical_tests.csv", index=False)
print("Saved results/fairness_statistical_tests.csv")

## How to read this

For each signal, look at:

- **wilcoxon_p_value**: smaller means the difference between original and counterfactual scores is unlikely to be just noise.
- **bootstrap_ci_low / bootstrap_ci_high**: if both are above zero, the average size of the score change is reliably positive.

Even with a small dataset, this is a much more honest way to talk about the fairness numbers than just reporting the mean.

In [ ]:
from google.colab import files
files.download("results/fairness_statistical_tests.csv")